# Late payment by large UK companies: a statistical analysis

**Patrick Gilliland**

**Question.** Is the share of invoices paid late lower in 2023 to 2024 than in 2018 to 2019, for the same large UK companies? By the share of invoices paid late I mean the data column `% Invoices not paid within agreed terms`.

**Data.** [Check when large businesses pay their suppliers](https://www.gov.uk/check-when-businesses-pay-invoices), the GOV.UK payment practices reports. Each row is one large company's report on how it paid its suppliers over a reporting period, and a large company must publish one at least twice a year. I use the export of all published reports from https://check-payment-practices.service.gov.uk/export/csv/, taken on 6 Oct 2026 and saved beside this notebook as `all_published_reports_20261006.csv`. Contains public sector information licensed under the Open Government Licence v3.0.

**Steps.** I check the file, load it without its free-text columns, describe it and list its anomalies before any analysis uses it. Then I build the analysis table from the Regime-1 reports of the paired companies, draw three visual models, train and evaluate the fitted model, run the hypothesis test and summarise what I found.

**Submission.** `analysis.ipynb`, the report as `Statistical_Analysis_Report.pdf` and `module_summary.pdf` (the same report under the two names the brief uses), `requirements.txt` and the dataset CSV.

## Setup

In [1]:
import hashlib
from pathlib import Path

import pandas as pd

# I show every column, whole cell text and two decimal places with thousands
# separators, so the wide tables below read without scientific notation.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", "{:,.2f}".format)

In [2]:
DATA = Path("all_published_reports_20261006.csv")
EXPORT_URL = "https://check-payment-practices.service.gov.uk/export/csv/"
EXPECTED_SHA256 = (
    "f9f2552222190b9c5f1ca145cdc1771ac88ba81c38fcfef49fe64c3b696db535"
)

if not DATA.exists():
    raise FileNotFoundError(
        f"{DATA} is not beside this notebook. It is the export of all "
        f"published reports from {EXPORT_URL}, taken on 6 Oct 2026, and a "
        "copy is in the submission zip. Save it here under this name."
    )
digest = hashlib.sha256(DATA.read_bytes()).hexdigest()
print(f"File: {DATA}, {DATA.stat().st_size:,} bytes")
print(f"SHA-256: {digest}")
# The export grows every day, so a new download is a different file and every
# figure below would change. I stop rather than analyse the wrong file.
if digest != EXPECTED_SHA256:
    raise ValueError(
        "This is not the export taken on 6 Oct 2026, which every figure in "
        "this notebook comes from. Use the copy in the submission zip."
    )
print("It matches the export taken on 6 Oct 2026.")

File: all_published_reports_20261006.csv, 103,043,357 bytes
SHA-256: f9f2552222190b9c5f1ca145cdc1771ac88ba81c38fcfef49fe64c3b696db535
It matches the export taken on 6 Oct 2026.


## Data handling

### Task 2: Load the dataset

In [3]:
# I take the date columns off the header, as the names that end in "date", so
# no column name is typed by hand. low_memory=False reads the file in one pass
# and gives each column one type; the whole file fits easily in memory here.
date_columns = [
    name
    for name in pd.read_csv(DATA, nrows=0).columns
    if name.endswith("date")
]
loaded = pd.read_csv(DATA, parse_dates=date_columns, low_memory=False)
print(f"As loaded: {len(loaded):,} rows, {loaded.shape[1]} columns")

# A text column whose median answer is longer than eight words holds written
# answers, not a code, a name or a yes/no. The analysis needs none of them, so
# I drop them here, before anything is shown, and never print one.
free_text = [
    name
    for name in loaded.select_dtypes(include="str").columns
    if loaded[name].str.split().str.len().median() > 8
]
df = loaded.drop(columns=free_text)
del loaded
print(f"Free-text columns dropped ({len(free_text)}):")
for name in free_text:
    print(f"  {name}")
print(f"Kept: {len(df):,} rows, {df.shape[1]} columns")

As loaded: 115,358 rows, 52 columns


Free-text columns dropped (10):
  Standard payment terms
  Payment terms have changed
  Suppliers notified of changes
  Maximum contractual payment period information
  Other information payment terms
  Description of specific circumstances for retention clauses
  Description of parity policy
  Retention clause money release description
  Description of stages for money release
  Dispute resolution process
Kept: 115,358 rows, 42 columns


In [4]:
# The flag columns hold only True, False or blank. Those with blanks load as
# object, so I give them pandas' nullable boolean type, which keeps all three.
flag_columns = [
    name
    for name in df.columns
    if df[name].dtype == object
    and set(df[name].dropna().unique()) <= {True, False}
]
df[flag_columns] = df[flag_columns].astype("boolean")
print(f"Flag columns converted from object to boolean: {len(flag_columns)}")

Flag columns converted from object to boolean: 11


In [5]:
df.head()

,Report Id,Policy Regime,Financial period start date,Start date,End date,Filing date,Company,Company number,Qualifying contracts in reporting period,Payments made in reporting period,Qualifying construction contracts in reporting period,Construction contracts have retention clauses,Average time to pay,Total value invoices paid within 30 days,Total value invoices paid between 31 and 60 days,Total value invoices paid later than 60 days,% Invoices paid within 30 days,% Invoices paid between 31 and 60 days,% Invoices paid later than 60 days,Total value invoices paid later than agreed terms,% Invoices not paid within agreed terms,% Invoices not paid due to dispute,Shortest (or only) standard payment period,Longest standard payment period,Maximum contractual payment period,Retention clauses included in all construction contracts,Retention clauses included in standard payment terms,Retention clauses are used in specific circumstances,Retention clauses used above a specific sum,Value above which retention clauses are used,Retention clauses are at a standard rate,Retention clauses standard rate percentage,Retention clauses have parity with client,Retention clause money release is staged,Retention value compared to client retentions as %,Retention value compared to total payments as %,Participates in payment codes,E-Invoicing offered,Supply-chain financing offered,Policy covers charges for remaining on supplier list,Charges have been made for remaining on supplier list,URL
0,2,Regime-1,NaT,2017-04-29,2017-10-28,2017-11-07,MEDTRONIC LIMITED,01070807,True,False,False,False,25.00,NaN,NaN,NaN,77.00,20.00,3.00,0.00,89.00,0.00,30.00,60.00,60.00,False,False,False,False,NaN,False,NaN,False,False,NaN,NaN,No,True,False,False,False,https://check-payment-practices.service.gov.uk/report/2
1,3,Regime-1,NaT,2017-05-01,2017-10-31,2017-11-08,SEBDEN STEEL SERVICE CENTRES LIMITED,02553464,True,False,False,False,69.00,NaN,NaN,NaN,7.00,28.00,65.00,0.00,0.00,0.00,30.00,NaN,75.00,False,False,False,False,NaN,False,NaN,False,False,NaN,NaN,No,False,False,False,False,https://check-payment-practices.service.gov.uk/report/3
2,4,Regime-1,NaT,2017-04-15,2017-10-14,2017-11-08,GREENERGY FLEXIGRID LIMITED,07581532,True,False,False,False,21.00,NaN,NaN,NaN,89.00,9.00,2.00,0.00,47.00,0.00,2.00,30.00,30.00,False,False,False,False,NaN,False,NaN,False,False,NaN,NaN,No,False,False,True,True,https://check-payment-practices.service.gov.uk/report/4
3,5,Regime-1,NaT,2017-04-15,2017-10-14,2017-11-08,GREENERGY FUELS LIMITED,04058825,True,False,False,False,14.00,NaN,NaN,NaN,90.00,8.00,2.00,0.00,31.00,0.00,2.00,30.00,30.00,False,False,False,False,NaN,False,NaN,False,False,NaN,NaN,No,False,False,True,True,https://check-payment-practices.service.gov.uk/report/5
4,6,Regime-1,NaT,2017-04-15,2017-10-14,2017-11-08,GREENERGY INTERNATIONAL LIMITED,02809935,True,False,False,False,15.00,NaN,NaN,NaN,96.00,2.00,2.00,0.00,14.00,0.00,2.00,30.00,30.00,False,False,False,False,NaN,False,NaN,False,False,NaN,NaN,No,False,False,True,False,https://check-payment-practices.service.gov.uk/report/6


In [6]:
print(f"Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
pd.DataFrame({"dtype": df.dtypes.astype(str), "blanks": df.isna().sum()})

Shape: 115,358 rows, 42 columns


,dtype,blanks
Report Id,int64,0
Policy Regime,str,0
Financial period start date,datetime64[us],95331
Start date,datetime64[us],0
End date,datetime64[us],0
Filing date,datetime64[us],0
Company,str,0
Company number,str,0
Qualifying contracts in reporting period,bool,0
Payments made in reporting period,bool,0


In [7]:
# I find each column the analysis uses by a phrase that matches its name and
# no other, so no name with a % sign or a space is typed in full by hand.
PHRASES = {
    "regime": "Regime",
    "company": "Company number",
    "start": "Start date",
    "end": "End date",
    "days": "Average time",
    "within_30": "Invoices paid within",
    "31_to_60": "Invoices paid between",
    "over_60": "Invoices paid later",
    "outcome": "not paid within agreed",
    "dispute": "due to dispute",
    "qualifying": "Qualifying contracts",
    "codes": "payment codes",
}
col = {}
for role, phrase in PHRASES.items():
    found = [name for name in df.columns if phrase in name]
    if len(found) != 1:
        raise KeyError(f"{phrase!r} matches {found}, not one column")
    col[role] = found[0]
pd.Series(col, name="column in the data").to_frame()

,column in the data
regime,Policy Regime
company,Company number
start,Start date
end,End date
days,Average time to pay
within_30,% Invoices paid within 30 days
31_to_60,% Invoices paid between 31 and 60 days
over_60,% Invoices paid later than 60 days
outcome,% Invoices not paid within agreed terms
dispute,% Invoices not paid due to dispute


**What loaded.** The file loads in one call as 115,358 reports and 52 columns. I drop the 10 free-text columns straight away, before anything is shown, which leaves every row and 42 columns. The four date columns parse as dates, the payment figures load as numbers, and `Company number` stays as text, which keeps its leading zeros. The 11 flag columns with blanks loaded as object, so I gave them pandas' nullable boolean type; the other four have no blanks and loaded as bool. `Report Id` is an identifier, not a measure.

**Blanks.** The payment figures are blank in 9,500 reports, and the share of invoices paid late in 9,423. `Financial period start date` is blank in 95,331, and the retention clause values and percentages are almost wholly blank; I use none of these. The anomaly list at the end of this section looks at the blanks the analysis depends on.

## Model definition

## Training

## Evaluation

## Summary